# 05 — Held-out evaluation and promotion gate

This notebook evaluates the exact checkpoint on a separate, provenance-checked validation manifest. Promotion is blocked until every modality/tool task has held-out coverage and approved numeric thresholds have been calibrated on the benchmark. A successful training run is not itself a merge or model release.

In [ ]:
import sys, json
from pathlib import Path

from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)

sys.path.insert(0, str(root / "src"))
from onecore.evaluate import evaluate_checkpoint, promotion_gate
from onecore.promote import promote_candidate

metrics = evaluate_checkpoint(
    root,
    root / "configs" / "phone-research.yaml",
    root / "run-state" / "checkpoint.pt",
    root / "run-state" / "tokenizer.json",
    root / "configs" / "promotion-thresholds.json",
    root / "run-state" / "evaluation.json",
)
passed, failures = promotion_gate(metrics)
print(json.dumps({"metrics": metrics, "promotion_gate_passed": passed, "failures": failures}, ensure_ascii=False, indent=2))
if passed:
    print(promote_candidate(root / "run-state" / "evaluation.json", root / "run-state" / "model-registry.json"))
else:
    print("PROMOTION BLOCKED. No candidate was merged or marked as released.")
